# QRT Challenge — Étude de Learning Curve

**Objectif** : Tracer l'évolution de l'accuracy en fonction de la taille du dataset d'entraînement pour 3 modèles :
- Régression Logistique
- XGBoost
- Random Forest

**Méthodologie** : Validation croisée stratifiée (3-fold) à chaque taille de données pour assurer la stabilité des résultats.

**Dataset** : `X_train.csv` + `y_train.csv` (données complètes)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

## 1. Chargement des données complètes

In [ ]:
X_raw = pd.read_csv('Data/X_train.csv')
y_raw = pd.read_csv('Data/y_train.csv')

df = X_raw.merge(y_raw, on='ROW_ID')
print(f'Dataset complet : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

## 2. Fonction de Prétraitement (`preprocess_data`)

- Création de la feature d'interaction `RET_1_TURNOVER`
- Remplissage des valeurs manquantes (0)
- Normalisation par groupe (StandardScaler intra-groupe)
- Création de la cible binaire (`target_SIGN`)

In [ ]:
def preprocess_data(df):
    """
    Pipeline complet de feature engineering et normalisation.
    
    Étapes :
    1. Feature interaction : RET_1 * MEDIAN_DAILY_TURNOVER
    2. Remplissage des NaN par 0
    3. Normalisation par groupe (StandardScaler intra-GROUP)
    4. Création de la cible binaire (sign of return)
    
    Returns:
        X (pd.DataFrame) : features normalisées
        y (pd.Series)    : cible binaire (0/1)
    """
    data = df.copy()
    
    # --- Feature Engineering ---
    data['RET_1_TURNOVER'] = data['RET_1'] * data['MEDIAN_DAILY_TURNOVER']
    
    # --- Target ---
    data['target_SIGN'] = (data['target'] > 0).astype(int)
    
    # --- Colonnes features ---
    exclude = ['ROW_ID', 'TS', 'ALLOCATION', 'target', 'target_SIGN']
    feature_cols = [c for c in data.columns if c not in exclude]
    
    # --- Remplissage des NaN ---
    X = data[feature_cols].fillna(0)
    y = data['target_SIGN']
    
    # --- Normalisation par groupe ---
    numeric_cols = [c for c in feature_cols if c != 'GROUP']
    for grp in X['GROUP'].unique():
        mask = X['GROUP'] == grp
        scaler = StandardScaler()
        X.loc[mask, numeric_cols] = scaler.fit_transform(X.loc[mask, numeric_cols])
    
    return X, y

# Prétraiter l'ensemble du dataset
X_full, y_full = preprocess_data(df)
print(f'Features : {X_full.shape[1]} colonnes')
print(f'Target distribution :\n{y_full.value_counts(normalize=True).to_string()}')

## 3. Définition des modèles

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, penalty='l2', solver='saga', max_iter=500, random_state=SEED
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=SEED,
        n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10,
        min_samples_split=5, min_samples_leaf=2,
        random_state=SEED, n_jobs=-1
    )
}

print('Modèles définis :', list(models.keys()))

## 4. Étude de Learning Curve

On évalue chaque modèle pour des tailles croissantes du dataset (de 5% à 100%).

Pour chaque point :
- Sous-échantillonnage stratifié du dataset
- **3-fold Stratified CV** pour mesurer l'accuracy
- On enregistre la moyenne et l'écart-type

In [ ]:
# --- Fractions du dataset à évaluer ---
fractions = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.0]

N_FOLDS = 3  # 3-fold CV (compromis vitesse/stabilité sur gros dataset)
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Stockage des résultats : {model_name: {'sizes': [], 'means': [], 'stds': []}}
lc_results = {name: {'sizes': [], 'means': [], 'stds': []} for name in models}

total_n = len(X_full)
print(f'Dataset total : {total_n:,} exemples')
print(f'Fractions à tester : {fractions}')
print(f'Tailles correspondantes : {[int(f * total_n) for f in fractions]}')
print(f'CV : {N_FOLDS}-fold stratifié\n')

for frac in fractions:
    n = int(frac * total_n)
    
    # Sous-échantillonnage stratifié
    if frac < 1.0:
        idx = np.arange(total_n)
        np.random.shuffle(idx)
        idx = idx[:n]
        X_sub = X_full.iloc[idx].reset_index(drop=True)
        y_sub = y_full.iloc[idx].reset_index(drop=True)
    else:
        X_sub = X_full
        y_sub = y_full
    
    print(f'--- n = {n:,} ({frac*100:.0f}%) ---')
    
    for name, model in models.items():
        t0 = time.time()
        scores = cross_val_score(model, X_sub, y_sub, cv=cv, scoring='accuracy', n_jobs=-1)
        elapsed = time.time() - t0
        
        lc_results[name]['sizes'].append(n)
        lc_results[name]['means'].append(scores.mean())
        lc_results[name]['stds'].append(scores.std())
        
        print(f'  {name:25s} → {scores.mean():.4f} ± {scores.std():.4f}  ({elapsed:.1f}s)')

print('\n✅ Learning curve computation done.')

## 5. Visualisation — Courbes d'apprentissage comparées

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = {'Logistic Regression': '#3498db', 'XGBoost': '#2ecc71', 'Random Forest': '#e74c3c'}
markers = {'Logistic Regression': 'o', 'XGBoost': 's', 'Random Forest': '^'}

for name, data in lc_results.items():
    sizes = np.array(data['sizes'])
    means = np.array(data['means'])
    stds = np.array(data['stds'])
    
    color = colors.get(name, 'gray')
    marker = markers.get(name, 'o')
    
    # Courbe principale
    ax.plot(sizes, means, marker=marker, label=name, color=color, linewidth=2, markersize=8)
    
    # Bande de confiance (±1 std)
    ax.fill_between(sizes, means - stds, means + stds, alpha=0.15, color=color)

# Ligne de base (random classifier)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.6, linewidth=1, label='Baseline (50%)')

ax.set_xlabel('Taille du dataset d\'entraînement (n)', fontsize=13)
ax.set_ylabel('Accuracy (CV)', fontsize=13)
ax.set_title('Learning Curve — Comparaison des modèles\n(3-Fold Stratified CV, dataset complet)', fontsize=15)
ax.legend(fontsize=12, loc='lower right')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=11)

# Formater l'axe X en milliers
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))

plt.tight_layout()
plt.savefig('learning_curve_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé : learning_curve_comparison.png')

## 6. Tableau récapitulatif

In [ ]:
# Résumé sous forme de DataFrame
rows = []
for name, data in lc_results.items():
    for i, size in enumerate(data['sizes']):
        rows.append({
            'Model': name,
            'n': size,
            'Accuracy (mean)': round(data['means'][i], 4),
            'Accuracy (std)': round(data['stds'][i], 4)
        })

summary_df = pd.DataFrame(rows)

# Pivot pour affichage clair
pivot = summary_df.pivot_table(index='n', columns='Model', values='Accuracy (mean)')
print('=== Accuracy par taille de dataset ===')
print(pivot.to_string())

# Quel modèle converge le plus vite ?
print('\n=== Performances à 100% des données ===')
for name, data in lc_results.items():
    print(f'{name:25s} → {data["means"][-1]:.4f} ± {data["stds"][-1]:.4f}')

## 7. Interprétation

- **Convergence rapide** = le modèle n'a pas besoin de plus de données → risque de sous-ajustement ou de feature engineering insuffisant.
- **Courbe montante** à 100% = le modèle bénéficierait de plus de données.
- **Plateau précoce** = ajouter des données n'apporte plus rien → se concentrer sur le feature engineering ou l'architecture du modèle.

### Actions possibles :
1. Si XGBoost >> Logistic Regression → les relations non-linéaires sont importantes
2. Si toutes les courbes plafonnent → investir dans de meilleures features
3. Si la courbe monte encore à 100% → récupérer plus de données ou augmenter les features